# Stage 1 — Non-Instruction Fine-Tuning (Domain Adaptation)

**Project:** Customer Support Assistant · **Base model:** TinyLlama-1.1B (via Unsloth, 4-bit / QLoRA)

Before teaching the model *how to answer*, we adapt it to the **domain language** of
customer support using raw, free-text paragraphs derived from the Kaggle
`suraj520/customer-support-ticket-dataset`.

This is *continued pretraining* style training: the model simply learns to predict
the next token over domain text (no instruction/response structure yet).

**Input:** `data/non_instruction_data.csv` (column `text`)
**Output:** LoRA adapter saved to `models/non_instruct_adapter/`

In [ ]:
# --- Environment setup (Google Colab, T4/A100 GPU) -----------------------
# Unsloth makes TinyLlama fine-tuning fit comfortably in a free Colab T4.
%%capture
# Install Unsloth and let it pull compatible recent transformers/trl/peft.
# (These notebooks use the CURRENT trl API: SFTConfig + processing_class.)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl peft accelerate bitsandbytes datasets
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# --- Get the project code + datasets into Colab -------------------------
# Option A: clone your GitHub repo (recommended once you have published it)
#   !git clone https://github.com/<your-username>/customer-support-assistant.git
#   %cd customer-support-assistant
#
# Option B: mount Google Drive and point PROJECT at your uploaded folder
#   from google.colab import drive; drive.mount('/content/drive')
#   PROJECT = '/content/drive/MyDrive/customer-support-assistant'
#
# For a local machine, just run the notebook from the project root.
import os
PROJECT = os.environ.get("PROJECT", ".")
DATA = os.path.join(PROJECT, "data")
MODELS = os.path.join(PROJECT, "models")
os.makedirs(MODELS, exist_ok=True)
print("PROJECT:", os.path.abspath(PROJECT))

## 1. Load TinyLlama in 4-bit with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/tinyllama"   # TinyLlama-1.1B, 4-bit ready

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # auto (fp16 on T4, bf16 on Ampere+)
    load_in_4bit = True,     # QLoRA
)

## 2. Attach LoRA adapters (QLoRA)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 3. Load the raw domain text dataset

In [ ]:
from datasets import load_dataset
import os

csv_path = os.path.join(DATA, "non_instruction_data.csv")
raw = load_dataset("csv", data_files=csv_path, split="train")
print(raw)
print("\nExample paragraph:\n", raw[0]["text"][:400])

EOS = tokenizer.eos_token
def add_eos(batch):
    return {"text": [t + EOS for t in batch["text"]]}
raw = raw.map(add_eos, batched=True)

## 4. Train (next-token prediction over domain text)

In [ ]:
from trl import SFTTrainer, SFTConfig

# NOTE (API compatibility): recent transformers renamed Trainer's `tokenizer`
# arg to `processing_class`, and recent trl moved dataset/text settings into
# SFTConfig. This cell uses that current API.
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,      # was: tokenizer = tokenizer
    train_dataset = raw,
    args = SFTConfig(                  # was: TrainingArguments
        dataset_text_field = "text",   # moved into SFTConfig
        max_seq_length = MAX_SEQ_LENGTH,   # if error: rename to max_length
        packing = True,                # pack short paragraphs together -> efficient
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,          # small experiment
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage1",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()
print(trainer_stats)

## 5. Quick sanity check — does it 'sound' like the domain now?

In [ ]:
FastLanguageModel.for_inference(model)
prompt = "A customer raised a billing ticket about"
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=60, temperature=0.7, do_sample=True)
print(tokenizer.batch_decode(out, skip_special_tokens=True)[0])

## 6. Save the domain-adapted adapter

In [ ]:
save_dir = os.path.join(MODELS, "non_instruct_adapter")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved non-instruction (domain-adapted) LoRA adapter to:", save_dir)

### Stage 1 done
We now have a TinyLlama adapter that has "read" a lot of customer-support text.
In **Stage 2** we continue from here (or from the base model) and teach it to
follow instructions using `data/instruction_dataset.jsonl`.